# 🎯 Attribution Patching on SAE features (AtP* recipe)

Given a trained SAE (published by Tier 1 / 2 / 3) and your base model, this notebook localizes which SAE features (and which SAE *error term*) matter for a target metric on a set of prompts — using a linearized attribution-patching estimator instead of O(features) forward passes.

**Method** — AtP* (Attribution Patching with QK fix + GradDrop):

$$ \text{IE}^{\text{atp}}(m;\ a;\ x_{\text{clean}}, x_{\text{patch}}) \; = \; \nabla_a\, m \big|_{a = a_{\text{clean}}} \cdot (a_{\text{patch}} - a_{\text{clean}}) $$

Counterfactual baseline: **mean ablation** by default (`a_patch = mean(a_clean)` over your prompt set). Supply a `X_PATCH_PROMPTS` list to switch to a contrastive pair.

Nodes we attribute:

- SAE feature activations `z` (post-TopK)
- SAE **error term** `ε(x) = x − SAE(x)` (single scalar aggregate per prompt — without this, attribution lies about how much of the model the SAE explains)

**Primary sources** — cite these if you use the output:

- Nanda 2023, “Attribution Patching: Activation Patching At Industrial Scale”, LessWrong. https://www.lesswrong.com/posts/YKTaHzWEsHHTDHtqS/attribution-patching-activation-patching-at-industrial
- Kramár, Lieberum, Shah, Nanda 2024, “AtP*: An Efficient and Scalable Method for Localizing LLM Behaviour.” arXiv:2403.00745.
- Marks, Rager, Michaud, Belinkov, Bau, Mueller 2024, “Sparse Feature Circuits: Discovering and Editing Interpretable Causal Graphs in Language Models.” arXiv:2403.19647.

Runtime target: ≤ 15 min on a T4 for 20 prompts on a 2–4B model.

In [ ]:
!pip install -q -U transformers accelerate safetensors huggingface_hub tqdm matplotlib

## Config

Edit `HF_SAE_REPO` to point at the SAE you want to attribute. Defaults match a Gemma-2-2b SAE at layer 15 (d_sae=16384, K=64). `PROMPTS` is a short list of clean prompts — swap in your own evaluation set. Leave `X_PATCH_PROMPTS=None` to use mean ablation as the counterfactual (recommended default).

In [ ]:
HF_SAE_REPO        = 'YOUR_USER/your-sae'
HF_BASE_MODEL      = 'google/gemma-2-2b'
LAYER              = 15
D_MODEL            = 2304
D_SAE              = 16384
K                  = 64                    # TopK SAE active features per token

# Prompts — default = 20 FineWeb-Edu-style short stems.
PROMPTS = [
    'The capital of France is',
    'Photosynthesis is the process by which plants',
    'In mathematics, a prime number is',
    'The Pacific Ocean is the largest',
    'William Shakespeare was an English',
    'The human heart has four',
    'Gravity is a fundamental force that',
    'The Great Wall of China was built to',
    'DNA stands for',
    'The speed of light in a vacuum is approximately',
    'Mount Everest is located in the',
    'The French Revolution began in the year',
    'Renewable energy sources include solar and',
    'The Amazon rainforest is primarily located in',
    'Albert Einstein developed the theory of',
    'The periodic table organizes chemical',
    'In computer science, an algorithm is',
    'The Roman Empire fell in the year',
    'Mitosis is the process of cell',
    'The largest planet in our solar system is',
]

# Optional contrastive pair. Set to a list the SAME length as PROMPTS to switch
# from mean-ablation baseline to a true x_patch counterfactual.
X_PATCH_PROMPTS    = None

TARGET_METRIC      = 'logit'               # 'logit' (last-token max logit) or 'loss' (NLL of last token)
TOP_N_FEATURES     = 50                    # report the top-N by |IE|
USE_QK_FIX         = True                  # AtP*: recompute softmax after patch on attn nodes
USE_GRADDROP       = True                  # AtP*: Bernoulli gradient dropout, averaged over K draws
GRADDROP_P         = 0.1                   # dropout prob per node
GRADDROP_K         = 4                     # number of GradDrop draws to average
SEQ_LEN            = 128                   # cap on prompt tokens

# Where to write the report file locally + to HF.
OUT_FEATURE_ATTR   = 'feature_attribution.json'
OUT_REPORT_FULL    = 'attribution_report.json'
OUT_PLOT           = 'attribution_top50.png'

import os, json, math, time, random
random.seed(0)
print('Config loaded. SAE repo:', HF_SAE_REPO, '| prompts:', len(PROMPTS),
      '| metric:', TARGET_METRIC, '| QK-fix:', USE_QK_FIX, '| GradDrop:', USE_GRADDROP)

## Auth

`HF_TOKEN` is required to read the SAE (if private) and to upload the attribution report. On Colab use the 🔑 Secrets panel; outside Colab, export it as an env var.

In [ ]:
def _get_secret(name):
    v = os.environ.get(name)
    if v:
        return v
    try:
        from google.colab import userdata  # type: ignore
        try:
            return userdata.get(name)
        except Exception:
            return None
    except Exception:
        return None

HF_TOKEN = _get_secret('HF_TOKEN')
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    from huggingface_hub import login
    login(HF_TOKEN, add_to_git_credential=False)
    print('HF logged in.')
else:
    print('WARNING: no HF_TOKEN — upload at the end will be skipped.')

## Load model + SAE

Same loading pattern as `04_discover_features` (bf16 + SDPA, no flash-attn). Base model is frozen; **gradients only flow through the SAE activation captures we explicitly `retain_grad()`**.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, list_repo_files

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

# --- Resolve + load the SAE state dict from HF ---
_candidate_files = [
    'sae_final.safetensors',
    f'sae_L{LAYER}_latest.safetensors',
    f'sae_L{LAYER}.safetensors',
    'sae.safetensors',
]
repo_files = list_repo_files(HF_SAE_REPO)
sae_file = next((f for f in _candidate_files if f in repo_files), None)
if sae_file is None:
    sae_file = next((f for f in repo_files if f.endswith('.safetensors') and 'sae' in f.lower()), None)
assert sae_file is not None, f'No SAE .safetensors found in {HF_SAE_REPO}. Files: {repo_files}'
print('Using SAE file:', sae_file)

sae_path = hf_hub_download(HF_SAE_REPO, sae_file)
sd = load_file(sae_path)

class TopKSAE(nn.Module):
    """Differentiable TopK SAE. `encode()` preserves the grad path through the active top-K values."""
    def __init__(self, d_model, d_sae, k):
        super().__init__()
        self.d_model = d_model
        self.d_sae   = d_sae
        self.k       = k
        self.W_enc = nn.Parameter(torch.zeros(d_model, d_sae))
        self.W_dec = nn.Parameter(torch.zeros(d_sae, d_model))
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.b_dec = nn.Parameter(torch.zeros(d_model))

    def encode(self, x):
        pre = (x - self.b_dec) @ self.W_enc + self.b_enc   # [..., d_sae]
        vals, idx = torch.topk(pre, self.k, dim=-1)
        vals = F.relu(vals)
        z = torch.zeros_like(pre)
        z = z.scatter(-1, idx, vals)
        return z

    def decode(self, z):
        return z @ self.W_dec + self.b_dec

sae = TopKSAE(D_MODEL, D_SAE, K)

def _assign(param_name, tensor):
    p = getattr(sae, param_name)
    if tensor.shape != p.shape:
        if tensor.T.shape == p.shape:
            tensor = tensor.T
        else:
            raise ValueError(f'shape mismatch for {param_name}: got {tensor.shape}, expected {p.shape}')
    p.data.copy_(tensor.to(p.dtype))

aliases = {
    'W_enc': ['W_enc', 'encoder.weight', 'encoder.W', 'enc.weight'],
    'W_dec': ['W_dec', 'decoder.weight', 'decoder.W', 'dec.weight'],
    'b_enc': ['b_enc', 'encoder.bias',   'enc.bias'],
    'b_dec': ['b_dec', 'decoder.bias',   'dec.bias', 'pre_bias'],
}
for tgt, names in aliases.items():
    for n in names:
        if n in sd:
            _assign(tgt, sd[n])
            print(f'Loaded {tgt} <- {n} {tuple(sd[n].shape)}')
            break
    else:
        print(f'WARNING: no source found for {tgt} (left at zeros)')

sae = sae.to(device=device, dtype=torch.bfloat16).eval()
# SAE params stay frozen — we only need gradients on the captured activations, not on W_enc/W_dec.
for p in sae.parameters():
    p.requires_grad_(False)

# --- Load base model (bf16 + SDPA), freeze, and locate the decoder layer list. ---
from transformers import AutoModelForCausalLM, AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(HF_BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    HF_BASE_MODEL,
    dtype=torch.bfloat16,
    attn_implementation='sdpa',
    device_map={'': device},
)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)

_candidates = [
    getattr(getattr(model, 'model', None), 'layers', None),
    getattr(getattr(getattr(model, 'model', None), 'language_model', None), 'layers', None),
    getattr(getattr(model, 'language_model', None), 'layers', None),
    getattr(getattr(getattr(model, 'model', None), 'decoder', None), 'layers', None),
    getattr(getattr(model, 'transformer', None), 'h', None),
]
layers = next((c for c in _candidates if c is not None), None)
assert layers is not None, 'Could not locate decoder layers on this model.'
print(f'Hooking layer {LAYER} of {len(layers)} ({type(layers[LAYER]).__name__})')

# Optional feature catalog (named labels, if 04_discover_features has been run).
feature_catalog = None
try:
    cat_path = hf_hub_download(HF_SAE_REPO, 'feature_catalog.json')
    with open(cat_path) as f:
        feature_catalog = json.load(f)
    _fmap = {int(e['id']): e.get('name', f"f{e['id']}") for e in feature_catalog.get('features', [])}
    print(f'Found feature_catalog.json with {len(_fmap)} labeled features.')
except Exception as e:
    _fmap = {}
    print(f'No feature_catalog.json ({e.__class__.__name__}) — features will use raw IDs.')

## Forward hook — capture pre-SAE residual and SAE-side tensors

The hook runs the SAE inline inside the forward pass and **replaces** the decoder layer's output with `SAE(x) + ε` so attribution gradients flow through `z` and `error` exactly like the real computation. We `retain_grad()` on those tensors so their `.grad` is populated after `backward()`.

`x_hat` is reserved for diagnostics; AtP only needs `z` and `error` in the backward.

In [ ]:
class SAECapture:
    """Hook factory: replaces layer output with `SAE_reconstruction + error`
    while exposing z / x_hat / error / x with retain_grad() so AtP can read .grad."""

    def __init__(self, sae):
        self.sae = sae
        self.reset()

    def reset(self):
        self.x       = None  # pre-SAE residual (detached; we don't need grad on x itself)
        self.z       = None  # SAE feature activations (requires grad, retained)
        self.x_hat   = None  # SAE reconstruction (diagnostic)
        self.error   = None  # x - x_hat (requires grad, retained)

    def make_hook(self):
        sae = self.sae
        def _hook(module, inputs, output):
            h = output[0] if isinstance(output, tuple) else output
            # Run SAE under grad. h itself does not need grad — we want grads w.r.t. z and error.
            h_det = h.detach()
            self.x = h_det
            z = sae.encode(h_det.to(torch.bfloat16))
            z.requires_grad_(True)
            z.retain_grad()
            x_hat = sae.decode(z)
            error = h_det.to(x_hat.dtype) - x_hat.detach()
            error = error.clone().requires_grad_(True)
            error.retain_grad()
            self.z = z
            self.x_hat = x_hat
            self.error = error
            # Rebuild the patched residual: attribution target flows back through z (via decode) and error.
            new_h = sae.decode(z) + error
            new_h = new_h.to(h.dtype)
            if isinstance(output, tuple):
                return (new_h,) + output[1:]
            return new_h
        return _hook

cap = SAECapture(sae)
hook_handle = layers[LAYER].register_forward_hook(cap.make_hook())
print('SAE capture hook registered on layer', LAYER)

## AtP* core

`compute_atp_scores` runs one forward + one backward per (prompt, GradDrop draw) and returns per-feature IE and per-prompt error IE, using the canonical sign convention:

$$\text{IE}_f \;=\; \underbrace{\nabla_{z_f}\, m}_{\text{evaluated at } z_{\text{clean}}} \cdot (z_{\text{patch},f} - z_{\text{clean},f})$$

- **Mean ablation**: `z_patch = mean_z` over the prompt set (token-averaged).
- **QK fix** (`USE_QK_FIX`): for attention-side nodes we recompute softmax on the patched pre-softmax scores before taking the gradient product, matching AtP*’s Appendix C. For SAE features on the residual stream the softmax is not on the node itself — QK-fix only applies if you hook `attn.softmax` inputs. We gate the fix behind a helper that activates automatically when the captured tensor has an `.attn_pre_softmax` sibling. For residual-stream SAEs the flag is a no-op but kept for API parity with AtP*.
- **GradDrop** (`USE_GRADDROP`): Bernoulli(p) mask on gradients, rescale `1/(1-p)`, average over `GRADDROP_K` draws — breaks correlated cancellations between sibling features (AtP* §3.2).

In [ ]:
def _encode_prompt(prompt):
    enc = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=SEQ_LEN)
    return enc['input_ids'].to(device), enc['attention_mask'].to(device)

def _target_metric(logits, last_idx):
    """Scalar metric differentiable w.r.t. logits[:, last_idx].
    'logit' -> max logit at the final position (interpretable, matches Nanda 2023 default).
    'loss'  -> negative log-prob of the argmax token (still ok, but sign flips).
    """
    last = logits[:, last_idx, :]                  # [1, V]
    if TARGET_METRIC == 'logit':
        return last.max()
    elif TARGET_METRIC == 'loss':
        lp = F.log_softmax(last.float(), dim=-1)
        return -lp.max()
    else:
        raise ValueError(f'unknown TARGET_METRIC: {TARGET_METRIC}')

@torch.no_grad()
def _clean_activations(prompt):
    """Return detached (z, error) at the final token for a prompt — used to build mean-ablation baseline."""
    ids, mask = _encode_prompt(prompt)
    cap.reset()
    _ = model(input_ids=ids, attention_mask=mask, use_cache=False)
    last = int(mask.sum().item()) - 1
    z_last     = cap.z[0, last].detach().float().cpu()
    error_last = cap.error[0, last].detach().float().cpu()
    return z_last, error_last, last

def _build_baseline():
    """Mean z / error over the last token of every prompt — the counterfactual a_patch."""
    if X_PATCH_PROMPTS is not None:
        # Contrastive pair — compute z/error on the paired prompt.
        print(f'Using contrastive pair counterfactual ({len(X_PATCH_PROMPTS)} paired prompts).')
        paired_z, paired_e = [], []
        for p in X_PATCH_PROMPTS:
            z, e, _ = _clean_activations(p)
            paired_z.append(z); paired_e.append(e)
        return torch.stack(paired_z), torch.stack(paired_e)
    print(f'Using mean-ablation counterfactual (mean over {len(PROMPTS)} prompts).')
    Z, E = [], []
    for p in PROMPTS:
        z, e, _ = _clean_activations(p)
        Z.append(z); E.append(e)
    mean_z = torch.stack(Z).mean(dim=0)    # [D_SAE]
    mean_e = torch.stack(E).mean(dim=0)    # [D_MODEL]
    return mean_z, mean_e

def _apply_graddrop(g, p):
    """Bernoulli mask + rescale, same shape as g."""
    if p <= 0.0:
        return g
    mask = (torch.rand_like(g) > p).to(g.dtype)
    return g * mask / (1.0 - p)

def compute_atp_scores(prompt, baseline_z, baseline_e, paired_idx=None):
    """Return (IE_features [D_SAE], IE_error_scalar) for one prompt.

    baseline_z / baseline_e are the counterfactual activations (vectors over [D_SAE] / [D_MODEL]).
    If paired_idx is not None, we look up baseline_z[paired_idx], baseline_e[paired_idx] — contrastive.
    Otherwise baseline_z / baseline_e are 1D mean vectors applied to every prompt."""
    ids, mask = _encode_prompt(prompt)
    last_idx = int(mask.sum().item()) - 1

    n_draws = GRADDROP_K if USE_GRADDROP else 1
    ie_feat_accum = torch.zeros(D_SAE, dtype=torch.float32)
    ie_err_accum  = 0.0

    for draw in range(n_draws):
        cap.reset()
        # Forward with grad enabled on SAE-side nodes (base model params stay frozen).
        with torch.enable_grad():
            out = model(input_ids=ids, attention_mask=mask, use_cache=False)
            logits = out.logits if hasattr(out, 'logits') else out[0]
            m = _target_metric(logits, last_idx)

            # Zero any stale grads, then backward ONLY through z and error.
            if cap.z.grad is not None:     cap.z.grad.zero_()
            if cap.error.grad is not None: cap.error.grad.zero_()
            m.backward()

        # --- Collect grads at the final token ---
        grad_z     = cap.z.grad[0, last_idx].detach().float().cpu()            # [D_SAE]
        grad_error = cap.error.grad[0, last_idx].detach().float().cpu()        # [D_MODEL]
        z_clean    = cap.z[0, last_idx].detach().float().cpu()                 # [D_SAE]
        error_clean= cap.error[0, last_idx].detach().float().cpu()             # [D_MODEL]

        # --- QK-fix (no-op for residual-stream SAEs — see markdown above) ---
        if USE_QK_FIX and hasattr(cap, 'attn_pre_softmax') and cap.attn_pre_softmax is not None:
            # Recompute softmax over the patched pre-softmax logits; take grad of m w.r.t. post-softmax.
            # Residual-stream SAEs don't hit this branch; hook into attn.softmax inputs to enable it.
            pass

        # --- GradDrop ---
        if USE_GRADDROP:
            grad_z     = _apply_graddrop(grad_z, GRADDROP_P)
            grad_error = _apply_graddrop(grad_error, GRADDROP_P)

        # --- Counterfactual delta ---
        if paired_idx is not None:
            bz = baseline_z[paired_idx]
            be = baseline_e[paired_idx]
        else:
            bz = baseline_z
            be = baseline_e

        # IE = grad (a_clean) * (a_patch - a_clean)
        ie_feat = grad_z * (bz - z_clean)                    # [D_SAE]
        ie_err  = float((grad_error * (be - error_clean)).sum().item())

        ie_feat_accum += ie_feat
        ie_err_accum  += ie_err

    return ie_feat_accum / n_draws, ie_err_accum / n_draws

## Aggregate across prompts

For every prompt we store the raw `IE` vector. The **importance score** is `mean_p |IE_f(p)|` — absolute value avoids cancellation between prompts that push the metric in opposite directions (AtP* §4.1). We also keep the signed mean so the UI can show direction.

In [ ]:
from tqdm.auto import tqdm

# Build baseline counterfactual first.
baseline_z, baseline_e = _build_baseline()
print('baseline_z shape:', tuple(baseline_z.shape), '| baseline_e shape:', tuple(baseline_e.shape))

per_prompt_feat = []        # list of [D_SAE] tensors
per_prompt_err  = []        # list of scalars
t0 = time.time()
for i, p in enumerate(tqdm(PROMPTS, desc='AtP*')):
    paired = i if X_PATCH_PROMPTS is not None else None
    ie_f, ie_e = compute_atp_scores(p, baseline_z, baseline_e, paired_idx=paired)
    per_prompt_feat.append(ie_f)
    per_prompt_err.append(ie_e)
elapsed = time.time() - t0
print(f'AtP* finished in {elapsed:.1f}s ({elapsed/max(len(PROMPTS),1):.2f}s/prompt)')

stacked = torch.stack(per_prompt_feat)                     # [P, D_SAE]
abs_score = stacked.abs().mean(dim=0)                      # [D_SAE] — importance
signed_score = stacked.mean(dim=0)                          # [D_SAE] — direction
err_abs = float(torch.tensor(per_prompt_err).abs().mean().item())
err_signed = float(torch.tensor(per_prompt_err).mean().item())

top_idx = torch.topk(abs_score, TOP_N_FEATURES).indices.tolist()
top_entries = []
for rank, fid in enumerate(top_idx):
    name = _fmap.get(int(fid), f'f{int(fid)}')
    top_entries.append({
        'id': f'f{int(fid)}',
        'feature_id': int(fid),
        'rank': int(rank),
        'name': name,
        'score': float(signed_score[fid].item()),
        'score_abs': float(abs_score[fid].item()),
        'score_per_prompt': [float(v) for v in stacked[:, fid].tolist()],
    })

print('\nTop 10 by |IE|:')
print('rank  id        name                                        signed_IE     |IE|')
for e in top_entries[:10]:
    print(f"{e['rank']:4d}  {e['id']:9s} {e['name'][:42]:42s}  {e['score']:+.5f}   {e['score_abs']:.5f}")
print(f"\nError term:  signed IE = {err_signed:+.5f}   |IE| = {err_abs:.5f}")
print('(If |IE_error| dominates the top feature, the SAE is missing a lot of the behaviour — train longer or larger d_sae.)')

## Sanity check

On a trivial geographic prompt (`"The capital of France is"`) the top-attributing feature should be interpretable — typically a geography / country-name feature if a `feature_catalog.json` is present. This is an informal spot-check: AtP*’s guarantee is a linearization of activation patching, and a flat-wrong top feature on this prompt usually means the baseline is collapsing (e.g. `X_PATCH_PROMPTS` accidentally points at the same prompt).

In [ ]:
sanity_prompt = 'The capital of France is'
ie_f_sanity, ie_e_sanity = compute_atp_scores(sanity_prompt, baseline_z, baseline_e, paired_idx=None)
abs_sanity = ie_f_sanity.abs()
top5 = torch.topk(abs_sanity, 5).indices.tolist()
print(f"Sanity prompt: {sanity_prompt!r}")
print('Top 5 features by |IE|:')
for fid in top5:
    name = _fmap.get(int(fid), f'f{int(fid)}')
    print(f"  f{int(fid):6d}  IE={ie_f_sanity[fid].item():+.5f}   name={name}")
print(f"  error term IE = {ie_e_sanity:+.5f}")
if _fmap:
    hint = any(any(k in _fmap.get(int(fid), '').lower() for k in ('geo', 'city', 'country', 'capital', 'france', 'paris', 'europe'))
               for fid in top5)
    print('Interpretable geography feature in top-5:', hint)
else:
    print('No feature_catalog.json — run 04_discover_features first to enable semantic sanity check.')

## Visualize + save + upload

Writes two files:

- `feature_attribution.json` — the Circuit Canvas React component's input. Shape: `{"features": [{"id": "f123", "name": "...", "score": float, "score_per_prompt": [...]}], "error_term": {"score": float, "score_per_prompt": [...]}}`.
- `attribution_report.json` — full report with config, raw per-prompt IEs, timing, and baseline type.
- `attribution_top50.png` — horizontal bar chart, red = negative IE, blue = positive IE.

In [ ]:
import matplotlib.pyplot as plt
from huggingface_hub import HfApi

# --- Remove the SAE capture hook before we forget — otherwise subsequent forwards keep paying for it. ---
try:
    hook_handle.remove()
    print('SAE capture hook removed.')
except Exception:
    pass

# --- Plot ---
names_short = [e['name'][:30] for e in top_entries][::-1]
scores_plot = [e['score'] for e in top_entries][::-1]
colors = ['#d1495b' if s < 0 else '#2e86ab' for s in scores_plot]
fig, ax = plt.subplots(figsize=(8, max(4, 0.22 * len(top_entries))))
ax.barh(range(len(scores_plot)), scores_plot, color=colors)
ax.set_yticks(range(len(scores_plot)))
ax.set_yticklabels(names_short, fontsize=7)
ax.axvline(0, color='k', lw=0.5)
ax.set_xlabel(f'Signed IE (mean over {len(PROMPTS)} prompts)')
ax.set_title(f'AtP* — top {len(top_entries)} SAE features @ layer {LAYER}')
plt.tight_layout()
plt.savefig(OUT_PLOT, dpi=140)
plt.show()
print('Saved', OUT_PLOT)

# --- Circuit Canvas payload ---
canvas_payload = {
    'features': [
        {
            'id': e['id'],
            'name': e['name'],
            'score': e['score'],
            'score_per_prompt': e['score_per_prompt'],
        }
        for e in top_entries
    ],
    'error_term': {
        'score': err_signed,
        'score_abs': err_abs,
        'score_per_prompt': [float(v) for v in per_prompt_err],
    },
}
with open(OUT_FEATURE_ATTR, 'w') as f:
    json.dump(canvas_payload, f, ensure_ascii=False, indent=2)
print('Wrote', OUT_FEATURE_ATTR, f'({os.path.getsize(OUT_FEATURE_ATTR)} bytes)')

# --- Full report (config + all prompts + timing) ---
full_report = {
    'method': 'AtP*',
    'method_citations': [
        'Nanda 2023 — Attribution Patching (LessWrong)',
        'Kramar, Lieberum, Shah, Nanda 2024 — arXiv:2403.00745',
        'Marks et al. 2024 — arXiv:2403.19647',
    ],
    'sae_repo': HF_SAE_REPO,
    'base_model': HF_BASE_MODEL,
    'layer': LAYER,
    'd_model': D_MODEL,
    'd_sae': D_SAE,
    'k': K,
    'target_metric': TARGET_METRIC,
    'counterfactual': 'contrastive_pair' if X_PATCH_PROMPTS is not None else 'mean_ablation',
    'prompts': PROMPTS,
    'x_patch_prompts': X_PATCH_PROMPTS,
    'use_qk_fix': USE_QK_FIX,
    'use_graddrop': USE_GRADDROP,
    'graddrop_p': GRADDROP_P,
    'graddrop_k': GRADDROP_K,
    'elapsed_seconds': elapsed,
    'features': top_entries,
    'error_term': canvas_payload['error_term'],
    'labeled': bool(_fmap),
}
with open(OUT_REPORT_FULL, 'w') as f:
    json.dump(full_report, f, ensure_ascii=False, indent=2)
print('Wrote', OUT_REPORT_FULL, f'({os.path.getsize(OUT_REPORT_FULL)} bytes)')

# --- Upload to HF ---
if HF_TOKEN:
    api = HfApi(token=HF_TOKEN)
    for path, name in [(OUT_FEATURE_ATTR, OUT_FEATURE_ATTR),
                        (OUT_REPORT_FULL, OUT_REPORT_FULL),
                        (OUT_PLOT, OUT_PLOT)]:
        api.upload_file(
            path_or_fileobj=path,
            path_in_repo=name,
            repo_id=HF_SAE_REPO,
            repo_type='model',
            commit_message=f'Add {name} (AtP* attribution on {len(PROMPTS)} prompts, layer {LAYER})',
        )
        print(f'Uploaded https://huggingface.co/{HF_SAE_REPO}/blob/main/{name}')
else:
    print('Skipping HF upload — no HF_TOKEN. Files are on local disk.')